# CardioIA — Fase 2 — Parte 1
## Extração de sintomas e sugestão de diagnóstico por regras

Este notebook implementa uma solução didática baseada em um mapa de conhecimento sintoma → possível diagnóstico. **Não é um sistema de diagnóstico clínico real.**

In [1]:
from pathlib import Path
import pandas as pd
import unicodedata
import re

def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

ARQ_FRASES = ROOT / 'data' / 'frases_pacientes.txt'
ARQ_MAPA = ROOT / 'data' / 'mapa_sintomas.csv'
print('Arquivos localizados em:', ROOT.resolve())

Arquivos localizados em: /mnt/data/CardioIA_Fase2


### 1. Leitura dos relatos e do mapa de conhecimento

In [2]:
frases = [linha.strip() for linha in ARQ_FRASES.read_text(encoding='utf-8').splitlines() if linha.strip()]
mapa = pd.read_csv(ARQ_MAPA)

print(f'Total de relatos: {len(frases)}')
print(f'Total de associações no mapa: {len(mapa)}')
mapa.head(10)

Total de relatos: 10
Total de associações no mapa: 31


,termo,sintoma_normalizado,diagnostico_possivel,peso
0,pressao no peito,dor ou pressão torácica,Síndrome coronariana aguda,3
1,dor no peito,dor ou pressão torácica,Síndrome coronariana aguda,2
2,dor irradiando para o braco esquerdo,irradiação da dor,Síndrome coronariana aguda,4
3,suor frio,sudorese,Síndrome coronariana aguda,3
4,nausea,náusea,Síndrome coronariana aguda,2
5,falta de ar,dispneia,Síndrome coronariana aguda,1
6,dor no peito,dor ou pressão torácica,Angina estável,2
7,aperto no peito,dor ou pressão torácica,Angina estável,3
8,pressao no peito,dor ou pressão torácica,Angina estável,2
9,esforco fisico,desencadeado por esforço,Angina estável,2


### 2. Regra de extração e pontuação
Cada expressão do mapa encontrada no relato gera uma pontuação para o diagnóstico associado. O diagnóstico com maior soma de pesos é apresentado como possibilidade.

In [3]:
mapa['termo_busca'] = mapa['termo'].apply(normalizar_texto)

def analisar_relato(relato, mapa_conhecimento):
    texto = normalizar_texto(relato)
    encontrados = []
    pontuacao = {}

    for _, linha in mapa_conhecimento.iterrows():
        if linha['termo_busca'] in texto:
            encontrados.append({
                'termo': linha['termo'],
                'sintoma': linha['sintoma_normalizado'],
                'diagnostico': linha['diagnostico_possivel'],
                'peso': int(linha['peso'])
            })
            diag = linha['diagnostico_possivel']
            pontuacao[diag] = pontuacao.get(diag, 0) + int(linha['peso'])

    if pontuacao:
        diagnostico = max(pontuacao, key=pontuacao.get)
        maior_pontuacao = pontuacao[diagnostico]
    else:
        diagnostico = 'Sem associação suficiente no mapa'
        maior_pontuacao = 0

    sintomas_unicos = sorted(set(item['sintoma'] for item in encontrados))
    return sintomas_unicos, diagnostico, maior_pontuacao, pontuacao


### 3. Aplicação aos 10 relatos

In [4]:
resultados = []

for i, relato in enumerate(frases, start=1):
    sintomas, diagnostico, score, scores = analisar_relato(relato, mapa)
    resultados.append({
        'paciente': i,
        'relato': relato,
        'sintomas_identificados': ', '.join(sintomas),
        'diagnostico_sugerido': diagnostico,
        'pontuacao': score
    })

df_resultados = pd.DataFrame(resultados)
pd.set_option('display.max_colwidth', 120)
df_resultados

,paciente,relato,sintomas_identificados,diagnostico_sugerido,pontuacao
0,1,"Paciente relata forte pressão no peito durante uma caminhada, com dor irradiando para o braço esquerdo e suor frio.","dor ou pressão torácica, irradiação da dor, sudorese",Síndrome coronariana aguda,10
1,2,"Paciente sente palpitações frequentes, tontura e teve um episódio de desmaio ao se levantar.","palpitações, síncope, tontura",Arritmia cardíaca,8
2,3,"Paciente apresenta falta de ar ao subir escadas, cansaço intenso e inchaço nas pernas ao final do dia.","dispneia, edema periférico, fadiga",Insuficiência cardíaca,10
3,4,"Paciente refere dor no peito ao fazer esforço físico, que melhora após alguns minutos de repouso.","desencadeado por esforço, dor ou pressão torácica, melhora com repouso",Angina estável,8
4,5,"Paciente relata dor de cabeça recorrente, tontura e visão turva em alguns momentos do dia.","alteração visual, cefaleia, tontura",Hipertensão arterial,6
5,6,Paciente acorda à noite com falta de ar e precisa dormir com vários travesseiros para respirar melhor.,"dispneia, dispneia paroxística noturna, ortopneia",Insuficiência cardíaca,10
6,7,Paciente sente batimentos cardíacos acelerados e irregulares acompanhados de fraqueza e tontura.,"fraqueza, ritmo irregular, taquicardia, tontura",Arritmia cardíaca,8
7,8,"Paciente apresenta náusea, suor frio, falta de ar e pressão intensa no centro do peito.","dispneia, dor ou pressão torácica, náusea, sudorese",Síndrome coronariana aguda,10
8,9,"Paciente relata cansaço progressivo, falta de ar mesmo em atividades leves e tornozelos inchados.","dispneia, edema periférico, fadiga, fadiga progressiva",Insuficiência cardíaca,11
9,10,"Paciente sente aperto no peito quando caminha rapidamente, mas o desconforto desaparece quando para para descansar.","desencadeado por esforço, dor ou pressão torácica, melhora com repouso",Angina estável,9


In [5]:
for r in resultados:
    print(f"Paciente {r['paciente']}: {r['diagnostico_sugerido']} | score={r['pontuacao']}")
    print(f"  Sintomas: {r['sintomas_identificados']}\n")

Paciente 1: Síndrome coronariana aguda | score=10
  Sintomas: dor ou pressão torácica, irradiação da dor, sudorese

Paciente 2: Arritmia cardíaca | score=8
  Sintomas: palpitações, síncope, tontura

Paciente 3: Insuficiência cardíaca | score=10
  Sintomas: dispneia, edema periférico, fadiga

Paciente 4: Angina estável | score=8
  Sintomas: desencadeado por esforço, dor ou pressão torácica, melhora com repouso

Paciente 5: Hipertensão arterial | score=6
  Sintomas: alteração visual, cefaleia, tontura

Paciente 6: Insuficiência cardíaca | score=10
  Sintomas: dispneia, dispneia paroxística noturna, ortopneia

Paciente 7: Arritmia cardíaca | score=8
  Sintomas: fraqueza, ritmo irregular, taquicardia, tontura

Paciente 8: Síndrome coronariana aguda | score=10
  Sintomas: dispneia, dor ou pressão torácica, náusea, sudorese

Paciente 9: Insuficiência cardíaca | score=11
  Sintomas: dispneia, edema periférico, fadiga, fadiga progressiva

Paciente 10: Angina estável | score=9
  Sintomas: desen

### Conclusão
A abordagem por regras consegue reconhecer expressões previamente cadastradas e produzir uma sugestão explicável. Sua principal limitação é depender do vocabulário do mapa e não compreender contexto clínico, negação ou ambiguidade como um modelo médico real.